# Track A — Fase 1 & 2: Cleanlab + Review Visual Semi-Otomatis
**BDC Satria Data 2026 — Klasifikasi Citra Sampah**

Notebook ini menjalankan:
- **Fase 1:** Cleanlab `find_label_issues()` → kandidat mislabel
- **Fase 2:** Filter sampel ambigu (margin/entropy rendah) + double-flagging
- **Review Visual:** Contact sheet semi-otomatis (grid 50 gambar/halaman)
- **Output:** `cleaning_log.csv` yang siap untuk Fase 3

> **Prasyarat:** Fase 0 (`03_oof_diagnosis.ipynb`) sudah selesai & `oof_diagnosis.csv` tersedia

---

In [ ]:
# ─── Cell 1: Setup ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
result = subprocess.run(
    ['git', 'clone', 'https://github.com/agaggigit/satria-data-bdcugm02.git', '/content/repo'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
sys.path.insert(0, '/content/repo')

!pip install -q cleanlab scipy seaborn scikit-learn
print('✅ Setup selesai')

In [ ]:
# ─── Cell 2: Konfigurasi Path ─────────────────────────────────────────────────
# ⚠️ SESUAIKAN path di bawah!

DRIVE_BASE    = '/content/drive/MyDrive/BDC2026 apace'
OOF_PATH      = f'{DRIVE_BASE}/output_trackB/oof.npy'
FOLDS_CSV     = f'{DRIVE_BASE}/output_trackA/folds.csv'
DIAG_CSV      = f'{DRIVE_BASE}/output_trackA/diagnosis/oof_diagnosis.csv'
DATASET_DIR   = f'{DRIVE_BASE}/BDC2026/train'  # folder train (untuk load gambar contact sheet)

OUTPUT_DIR    = f'{DRIVE_BASE}/output_trackA'
CONTACT_DIR   = f'{OUTPUT_DIR}/contact_sheets'

import os
os.makedirs(CONTACT_DIR, exist_ok=True)

print('Path dikonfigurasi:')
for name, path in [('OOF', OOF_PATH), ('folds.csv', FOLDS_CSV), ('diagnosis.csv', DIAG_CSV)]:
    status = '✅' if os.path.exists(path) else '❌ TIDAK ADA'
    print(f'  {status} {name}: {path}')

In [ ]:
# ─── Cell 3: Load Data ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

oof      = np.load(OOF_PATH)
folds_df = pd.read_csv(FOLDS_CSV)
diag_df  = pd.read_csv(DIAG_CSV)

print(f'OOF shape  : {oof.shape}')
print(f'folds rows : {len(folds_df):,}')
print(f'diag rows  : {len(diag_df):,}')
print(f'\nCek alignment: {(diag_df["filepath"] == folds_df["filepath"]).all()}')

---
## Fase 1: Cleanlab — Temukan Kandidat Mislabel

In [ ]:
# ─── Cell 4: Jalankan Cleanlab ────────────────────────────────────────────────
from track_a.src.cleanlab_runner import run_full_cleanlab

label_issues_df = run_full_cleanlab(
    oof            = oof,
    folds_df       = folds_df,
    diag_df        = diag_df,
    output_dir     = OUTPUT_DIR,
    use_cleanlab   = True,    # set False jika cleanlab gagal install
    frac_noise     = 0.10,    # asumsi 10% noise — bisa dituning
)

print(f'\nCandidates dari cleanlab: {len(label_issues_df):,}')
label_issues_df.head(10)

---
## Fase 2: Filter Sampel Ambigu

In [ ]:
# ─── Cell 5: Filter Ambigu ────────────────────────────────────────────────────
from track_a.src.ambiguous_filter import run_full_ambiguous_filter

ambiguous_df = run_full_ambiguous_filter(
    diag_df       = diag_df,
    folds_df      = folds_df,
    output_dir    = OUTPUT_DIR,
    label_issues_df = label_issues_df,   # untuk double-flagging
    auto_threshold_percentile = 5.0,     # ambil 5% paling tidak yakin
    total_train   = 26527,
)

print(f'\nTotal kandidat ambigu: {len(ambiguous_df):,}')
ambiguous_df.head(10)

In [ ]:
# ─── Cell 6: Gabung label_issues + ambiguous ──────────────────────────────────
# Gabungkan kedua daftar untuk review visual: union dari cleanlab + ambiguous filter
# Double-flagged (muncul di keduanya) = prioritas utama

all_candidates = pd.concat([
    label_issues_df.assign(source='cleanlab'),
    ambiguous_df.assign(source='ambiguous_filter'),
], ignore_index=True)

# Deduplikasi berdasarkan filepath
all_candidates = all_candidates.drop_duplicates(subset='filepath', keep='first')

# Prioritas: double_flagged → priority_pair → margin ascending
sort_cols = []
if 'is_double_flagged' in all_candidates.columns:
    sort_cols.append(('is_double_flagged', False))
if 'is_priority_pair' in all_candidates.columns:
    sort_cols.append(('is_priority_pair', False))
sort_cols.append(('margin', True))

sort_by = [c for c, _ in sort_cols]
sort_asc = [a for _, a in sort_cols]
all_candidates = all_candidates.sort_values(sort_by, ascending=sort_asc).reset_index(drop=True)

print(f'Total kandidat gabungan (dedup): {len(all_candidates):,}')
all_candidates[['filepath', 'label', 'pred_class', 'margin', 'source']].head(15)

---
## Review Visual: Contact Sheet Semi-Otomatis

Grid gambar 50 per halaman:
- 🔴 **Border merah** = label asli ≠ prediksi (probable mislabel)
- 🟠 **Border oranye** = benar tapi margin rendah (gambar ambigu)
- 🟢 **Border hijau** = benar & margin normal

Caption: `label_asli→prediksi | M=margin | Q=quality_score`

**Cara review:**
1. Lihat contact sheet per pair kelas
2. Putuskan per batch: kebanyakan `drop`? `relabel`? `keep`?
3. Catat di `cleaning_log.csv` (diinisialisasi di bawah)

In [ ]:
# ─── Cell 7: Generate Contact Sheets ─────────────────────────────────────────
from track_a.src.contact_sheet import generate_contact_sheets

# CATATAN: untuk Colab, gambar harus bisa diakses dari path di Drive
# Pastikan DATASET_DIR sudah benar!

figs = generate_contact_sheets(
    candidates           = all_candidates,
    output_dir           = CONTACT_DIR,
    group_by_pair        = True,      # kelompokkan per pasangan kelas
    n_per_page           = 50,        # 50 gambar per halaman
    n_cols               = 10,        # 10 kolom = 5 baris
    img_size             = 128,       # kecilkan untuk Colab cepat
    save_png             = True,      # simpan ke Drive
    display_in_notebook  = True,      # tampilkan di sini
    max_pages            = None,      # None = semua halaman (batasi jika terlalu banyak)
)

print(f'\n{len(figs)} halaman contact sheet selesai!')
print(f'PNG tersimpan di: {CONTACT_DIR}')

In [ ]:
# ─── Cell 8: Inisialisasi Cleaning Log ───────────────────────────────────────
from track_a.src.contact_sheet import init_cleaning_log

LOG_PATH = f'{OUTPUT_DIR}/cleaning_log.csv'

log = init_cleaning_log(all_candidates, output_path=LOG_PATH)
print(f'\ncleaning_log.csv siap di: {LOG_PATH}')
log.head(5)

---
## 📝 Cara Mengisi cleaning_log.csv

Download `cleaning_log.csv` dari Drive, edit, lalu upload kembali.

Isi kolom:
| Kolom | Nilai Valid | Keterangan |
|-------|-------------|------------|
| `keputusan` | `keep` / `relabel` / `drop` | **Wajib diisi** |
| `label_baru` | 0, 1, atau 2 | Wajib jika `keputusan=relabel`; sama dengan `label_asli` jika `keep/drop` |
| `alasan` | teks bebas | Singkat: "foto laptop berlabel Organic" |
| `reviewer` | nama/inisial | Untuk audit |

**Tips:**
- Lihat contact sheet per pair, putuskan per kelompok
- Relabel > drop jika kelas yang benar jelas terlihat
- Drop hanya jika gambar benar-benar tidak bisa diputuskan
- Total drop ≤ 2–3% dari train (~500–800 gambar)

In [ ]:
# ─── Cell 9: Validasi Log Setelah Review ─────────────────────────────────────
# Jalankan ini setelah selesai mengisi cleaning_log.csv!

from track_a.src.contact_sheet import validate_cleaning_log

log_validated = validate_cleaning_log(LOG_PATH)
print('\n→ Jika semua OK, lanjutkan ke 05_generate_v2.ipynb')

---

## ✅ Fase 1 & 2 Selesai!

Output di Drive:
- `label_issues.csv` — kandidat mislabel dari cleanlab
- `ambiguous_candidates.csv` — sampel ambigu (margin/entropy rendah)
- `contact_sheets/` — grid PNG untuk review visual
- `cleaning_log.csv` — log keputusan review (isi manual!)

**Next step:** Isi `cleaning_log.csv`, lalu jalankan `05_generate_v2.ipynb`.
